# Modern Deep Learning for Computer Vision: TensorFlow/Keras vs PyTorch

**DeepVision-Masterclass · Modern dual-framework edition**

This notebook modernizes the repository's image-classification workflow while preserving the original notebooks. It implements the same compact CNN experiment in **TensorFlow/Keras** and **PyTorch**, using the same MNIST train/validation/test split and comparable preprocessing.

### Learning outcomes
- Inspect the software environment and available CPU/GPU devices.
- Build reproducible input pipelines in both frameworks.
- Implement and train equivalent convolutional neural networks.
- Evaluate both models with accuracy, precision, recall, F1-score and confusion matrices.
- Compare training time and test performance without treating one run as a definitive benchmark.
- Reuse the data-loading pattern with the repository's optional Homer/Bart image folders.

> **Compatibility:** Python 3.10+. Install current mutually compatible TensorFlow/Keras and PyTorch/torchvision builds for your OS and accelerator. Use the official [TensorFlow install guide](https://www.tensorflow.org/install) and [PyTorch selector](https://pytorch.org/get-started/locally/). This notebook prints the exact versions used; it deliberately avoids hard-coding a potentially incompatible CUDA wheel.

## 1. Environment setup

Recommended: create a fresh virtual environment, then install Jupyter, TensorFlow, PyTorch and the utilities below. Install PyTorch/torchvision using the official selector first so the wheel matches your CPU/CUDA/ROCm setup.

```bash
python -m pip install --upgrade pip
python -m pip install jupyterlab tensorflow scikit-learn matplotlib pandas pillow
# Install torch + torchvision from https://pytorch.org/get-started/locally/
```

On Google Colab, the frameworks are usually preinstalled. Restart the runtime after changing major framework versions.

In [ ]:
import os
import random
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__ if hasattr(keras, "__version__") else "bundled with TensorFlow")
print("PyTorch:", torch.__version__)
try:
    import torchvision
    print("Torchvision:", torchvision.__version__)
except Exception as exc:
    print("Torchvision import issue:", repr(exc))

print("TensorFlow devices:", tf.config.list_physical_devices())
print("PyTorch device:", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

## 2. Dataset: MNIST

MNIST is already used by the repository's original GAN material. We reuse it here so the two frameworks can be compared on a dataset already represented in the masterclass.

The official Keras loader downloads the dataset if it is not cached. The same NumPy arrays are then passed to both frameworks, avoiding differences caused by separate dataset downloads or split logic.

In [ ]:
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 3  # Increase to 5-10 for a longer teaching experiment

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

# Deterministic validation split from the original training partition.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_full))
val_idx, train_idx = indices[:5000], indices[5000:]

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)
print("Classes:", np.unique(y_train))

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for digit, ax in enumerate(axes):
    sample_idx = np.where(y_train == digit)[0][0]
    ax.imshow(x_train[sample_idx], cmap="gray")
    ax.set_title(str(digit))
    ax.axis("off")
plt.suptitle("One training example per class")
plt.tight_layout()
plt.show()

## 3. Shared preprocessing and data pipelines

Both models receive grayscale images normalized to [0, 1]. TensorFlow uses NHWC layout `(batch, height, width, channels)`; PyTorch uses NCHW layout `(batch, channels, height, width)`. Labels remain integer class IDs and both models use sparse/cross-entropy classification loss.

In [ ]:
def tf_images(images):
    images = images.astype("float32") / 255.0
    return images[..., np.newaxis]  # NHWC

x_train_tf = tf_images(x_train)
x_val_tf = tf_images(x_val)
x_test_tf = tf_images(x_test)

tf_train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train_tf, y_train))
    .shuffle(len(x_train_tf), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
tf_val_ds = tf.data.Dataset.from_tensor_slices((x_val_tf, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
tf_test_ds = tf.data.Dataset.from_tensor_slices((x_test_tf, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("TensorFlow batch:", next(iter(tf_train_ds))[0].shape)

In [ ]:
def torch_images(images):
    # NCHW float32 tensor in [0, 1].
    array = images.astype("float32") / 255.0
    return torch.from_numpy(array[:, None, :, :])

torch_train_ds = TensorDataset(torch_images(x_train), torch.from_numpy(y_train.astype("int64")))
torch_val_ds = TensorDataset(torch_images(x_val), torch.from_numpy(y_val.astype("int64")))
torch_test_ds = TensorDataset(torch_images(x_test), torch.from_numpy(y_test.astype("int64")))

torch_device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
pin_memory = torch_device.type == "cuda"

torch_train_loader = DataLoader(torch_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_memory)
torch_val_loader = DataLoader(torch_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)
torch_test_loader = DataLoader(torch_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)

print("PyTorch batch:", next(iter(torch_train_loader))[0].shape)
print("Training device:", torch_device)

## 4. TensorFlow/Keras CNN

Architecture: two convolution + max-pooling blocks, a 128-unit dense layer, dropout, and a 10-class logits layer. The model returns logits; `SparseCategoricalCrossentropy(from_logits=True)` is therefore used.

In [ ]:
def build_tf_model():
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    logits = layers.Dense(NUM_CLASSES, name="logits")(x)
    return keras.Model(inputs, logits, name="mnist_cnn_tensorflow")

tf_model = build_tf_model()
tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
tf_model.summary()

In [ ]:
tf_start = time.perf_counter()
tf_history = tf_model.fit(
    tf_train_ds,
    validation_data=tf_val_ds,
    epochs=EPOCHS,
    verbose=1
)
tf_training_seconds = time.perf_counter() - tf_start

tf_test_loss, tf_test_accuracy = tf_model.evaluate(tf_test_ds, verbose=0)
print(f"TensorFlow test accuracy: {tf_test_accuracy:.4f}")
print(f"TensorFlow training time: {tf_training_seconds:.1f} seconds")

## 5. PyTorch CNN

This implementation mirrors the Keras architecture. PyTorch's `CrossEntropyLoss` expects raw logits and integer class labels, so no softmax is applied in the model.

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch_model = MNISTCNN().to(torch_device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)
print(torch_model)

In [ ]:
def run_torch_epoch(model, loader, training):
    model.train(training)
    total_loss, total_correct, total_examples = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(torch_device, non_blocking=True)
        labels = labels.to(torch_device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

torch_history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
torch_start = time.perf_counter()

for epoch in range(EPOCHS):
    train_loss, train_acc = run_torch_epoch(torch_model, torch_train_loader, training=True)
    val_loss, val_acc = run_torch_epoch(torch_model, torch_val_loader, training=False)
    torch_history["loss"].append(train_loss)
    torch_history["accuracy"].append(train_acc)
    torch_history["val_loss"].append(val_loss)
    torch_history["val_accuracy"].append(val_acc)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

torch_training_seconds = time.perf_counter() - torch_start
torch_test_loss, torch_test_accuracy = run_torch_epoch(torch_model, torch_test_loader, training=False)
print(f"PyTorch test accuracy: {torch_test_accuracy:.4f}")
print(f"PyTorch training time: {torch_training_seconds:.1f} seconds")

## 6. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tf_history.history["loss"], label="TensorFlow train")
axes[0].plot(tf_history.history["val_loss"], label="TensorFlow validation")
axes[0].plot(torch_history["loss"], "--", label="PyTorch train")
axes[0].plot(torch_history["val_loss"], "--", label="PyTorch validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(tf_history.history["accuracy"], label="TensorFlow train")
axes[1].plot(tf_history.history["val_accuracy"], label="TensorFlow validation")
axes[1].plot(torch_history["accuracy"], "--", label="PyTorch train")
axes[1].plot(torch_history["val_accuracy"], "--", label="PyTorch validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Test-set evaluation

We report accuracy, macro precision, macro recall and macro F1. Macro averaging gives each class equal weight. The test set is held out from model fitting and validation-based decisions.

In [ ]:
tf_probabilities = tf.nn.softmax(tf_model.predict(x_test_tf, batch_size=BATCH_SIZE, verbose=0), axis=1).numpy()
tf_predictions = tf_probabilities.argmax(axis=1)

torch_model.eval()
torch_predictions = []
with torch.inference_mode():
    for images, _ in torch_test_loader:
        logits = torch_model(images.to(torch_device))
        torch_predictions.extend(logits.argmax(dim=1).cpu().numpy())
torch_predictions = np.asarray(torch_predictions)

def classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro precision": precision,
        "Macro recall": recall,
        "Macro F1": f1,
    }

results = pd.DataFrame({
    "TensorFlow/Keras": classification_metrics(y_test, tf_predictions),
    "PyTorch": classification_metrics(y_test, torch_predictions),
}).T
results["Training time (s)"] = [tf_training_seconds, torch_training_seconds]
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, tf_predictions)).plot(
    ax=axes[0], colorbar=False, values_format="d"
)
axes[0].set_title("TensorFlow/Keras")
ConfusionMatrixDisplay(confusion_matrix(y_test, torch_predictions)).plot(
    ax=axes[1], colorbar=False, values_format="d"
)
axes[1].set_title("PyTorch")
plt.tight_layout()
plt.show()

print("TensorFlow/Keras report")
print(classification_report(y_test, tf_predictions, digits=4, zero_division=0))
print("PyTorch report")
print(classification_report(y_test, torch_predictions, digits=4, zero_division=0))

## 8. Save trained models

Weights are saved locally under `checkpoints/`. These generated files should normally remain outside Git unless you intentionally publish small, appropriately licensed artifacts.

In [ ]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

tf_model.save_weights(checkpoint_dir / "mnist_cnn_tensorflow.weights.h5")
torch.save(
    {
        "model_state_dict": torch_model.state_dict(),
        "class_count": NUM_CLASSES,
        "seed": SEED,
        "torch_version": torch.__version__,
    },
    checkpoint_dir / "mnist_cnn_pytorch.pt"
)
print("Saved checkpoints to:", checkpoint_dir.resolve())

## 9. Optional extension: Homer/Bart custom image folders

The original classification and transfer-learning notebooks refer to Homer/Bart image archives. These archives are **not bundled** with the repository, and the original private Drive archive's public provenance is not confirmed. Do not assume a similarly named Kaggle dataset is identical; verify the archive contents, class mapping, and license before use.

Expected folder layout:

```text
data/homer_bart/
├── train/
│   ├── bart/
│   └── homer/
├── validation/
│   ├── bart/
│   └── homer/
└── test/
    ├── bart/
    └── homer/
```

The next cell demonstrates a TensorFlow loader. The following one demonstrates the equivalent PyTorch `ImageFolder` loader. Run them only after preparing your own licensed dataset.

In [ ]:
HOMER_BART_ROOT = Path("data/homer_bart")
if (HOMER_BART_ROOT / "train").exists():
    hb_train_tf = keras.utils.image_dataset_from_directory(
        HOMER_BART_ROOT / "train", image_size=(224, 224), batch_size=32,
        label_mode="int", seed=SEED
    )
    hb_val_tf = keras.utils.image_dataset_from_directory(
        HOMER_BART_ROOT / "validation", image_size=(224, 224), batch_size=32,
        label_mode="int", seed=SEED, shuffle=False
    )
    hb_class_names = hb_train_tf.class_names
    hb_train_tf = hb_train_tf.prefetch(tf.data.AUTOTUNE)
    hb_val_tf = hb_val_tf.prefetch(tf.data.AUTOTUNE)
    print("TensorFlow classes:", hb_class_names)
else:
    print("Optional Homer/Bart dataset not found. Expected:", HOMER_BART_ROOT.resolve())

In [ ]:
if (HOMER_BART_ROOT / "train").exists():
    from torchvision import datasets, transforms

    hb_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    hb_train_pt = datasets.ImageFolder(HOMER_BART_ROOT / "train", transform=hb_transform)
    hb_val_pt = datasets.ImageFolder(HOMER_BART_ROOT / "validation", transform=hb_transform)

    assert hb_train_pt.classes == hb_val_pt.classes, "Train/validation class order differs."
    hb_train_loader = DataLoader(hb_train_pt, batch_size=32, shuffle=True, num_workers=0)
    hb_val_loader = DataLoader(hb_val_pt, batch_size=32, shuffle=False, num_workers=0)
    print("PyTorch classes:", hb_train_pt.classes)
else:
    print("Skipped: optional Homer/Bart folders are not available.")

## 10. Interpretation and experiment checklist

Before comparing or reporting results:
1. Run the notebook from top to bottom in a clean environment.
2. Record framework versions, device, batch size, epochs, seed and preprocessing.
3. Repeat the experiment with several seeds; a single run is not a robust benchmark.
4. For fair timing, use the same hardware and account for GPU warm-up and asynchronous execution. This introductory notebook reports wall-clock training time only.
5. Tune both implementations comparably and verify that their architectures, initialization, data order and optimization settings are aligned before drawing performance conclusions.
6. For Homer/Bart, report the exact dataset source, license, class counts, split strategy and any data-cleaning steps.

### Further exercises
- Add data augmentation (random crop/flip/rotation) to both pipelines.
- Compare SGD with momentum against Adam.
- Add early stopping, learning-rate scheduling and TensorBoard logging.
- Replace the small CNN with ResNet18/ResNet50 transfer learning in both frameworks.
- Add per-class metrics and calibration analysis.
- Export TensorFlow SavedModel and PyTorch TorchScript/ONNX, then compare inference latency.

### Dataset and model provenance
- **MNIST:** official dataset loader via `tf.keras.datasets.mnist`; originally used by the repository's GAN notebook.
- **Homer/Bart:** optional external image-folder dataset referenced by original classification/transfer-learning materials; source and redistribution rights must be independently verified.
- **Pretrained models:** if you extend this notebook with pretrained weights, cite the model source and follow its license/terms.

**This notebook is an educational framework comparison, not a claim that one framework is universally faster or more accurate.**